# Build a Multi-User Conversational Agentic AI System with LangGraph


## Problem Statement

We aim to build **HealthBuddy**, an Agentic AI system designed to assist users with health-related queries through intelligent reasoning and tool use. Leveraging the **ReAct framework** (Reasoning + Acting), HealthBuddy integrates a **Large Language Model (LLM)** with custom tools and structured instructions to provide accurate, actionable, and personalized responses.

HealthBuddy is capable of:

- Answering health-related queries, including symptom checks and treatment suggestions, by using external sources such as **Web Search** and **PubMed Search** tools
- Recommending appropriate doctors for specific medical concerns via a **Doctor Recommendation Tool**
- Displaying available **appointment slots** and **booking appointments** after collecting essential user details (name, email, phone)
- Maintaining **multi-turn conversations** in a natural, chat-like flow
- Supporting **multi-user interactions** with isolated memory contexts for each user to enable personalized and continuous engagement

Each notebook in this series will build and extend different components of the system, progressively enhancing HealthBuddy’s intelligence, tool-use capabilities, and user experience.


### Objective

Our goals in this notebook are:

- Enable **memory and multi-turn conversation support** for a tool-using agent

- Simulate a **realistic healthcare assistant** capable of live information retrieval, doctor recommendations, and appointment management

- Support **multi-user sessions** by maintaining user-specific memory with LangGraph

- Build and connect **specialized tools**, including:

    - `search_web`, `pubmed_search` for external data

    - `recommend_doctor`, `list_all_doctors` for doctor guidance

    - `show_available_slots`, `book_appointment` for scheduling

- Leverage **LangGraph and LangChain** to orchestrate reasoning, tool use, and memory

### Agent Architecture

The following diagram shows the HealthBuddy agent architecture:

- Users send conversational queries to the system.

- The agent, powered by OpenAI GPT-4o-mini, uses a ReAct-style loop to:

    - Analyze the query

    - Determine if a tool is needed

    - Call the appropriate tool

    - Observe results

    - Continue reasoning or finalize a response

- Tools include:

    - `search_web` and `pubmed_search` for symptom/treatment lookups

    - `recommend_doctor` and `list_all_doctors` to find specialists

    - `show_available_slots` to view open appointments

    - `book_appointment` to confirm bookings

- Interactions use simulated internal data structures (replaceable with external databases).

- User-specific memory enables multi-turn conversations without repetition.

- Concurrent multi-user support ensures isolated, personalized experiences.


In [ ]:
from IPython.display import Image
Image(filename='/Users/csharm33/code/genai_dbx/Course4/Data/conversational_agent.png')

## Load Necessary Dependencies


In [ ]:
import json
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain_core.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import display, Image, Markdown
from langgraph.checkpoint.memory import MemorySaver


## Setup Authentication and LLM Client

The following code sets up the UAIS environment to authenticate the application and securely connect to the Azure OpenAI services. It retrieves the access token required for authorization and initializes both the LLM and embedding clients using Azure OpenAI authentication.


In [ ]:
import os
from dotenv import load_dotenv
import httpx

def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course4/Data/vars.env')


AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
OPENAI_API_VERSION = os.environ["OPENAI_API_VERSION"]
EMBEDDINGS_DEPLOYMENT_NAME = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]
MODEL_DEPLOYMENT_NAME = os.environ["MODEL_DEPLOYMENT_NAME"]
PROJECT_ID = os.environ['PROJECT_ID']
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")

chat_client = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment = MODEL_DEPLOYMENT_NAME,
    temperature=0,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": PROJECT_ID
    }
)


embeddings_client = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": PROJECT_ID
    }
)


## Prepare Databases for Simulated Web Search and PubMed Search Tools

Here, we will load sample data simulating real web page documents and research papers from PubMed.

Our tools, which we will define later, will reference this data instead of making live internet searches (due to environment and security restrictions preventing internet access).

This simulates real-time web or research paper searches using preloaded documents as the source.

We will create two separate vector databases:

- `web_search_db`: Contains general-purpose health-related web articles, simulating responses to real-time internet queries like “treatments for diabetes” or “how to handle panic attacks”

- `pubmed_db`: Built using sample abstracts and content from medical research papers, simulating PubMed search results for clinically accurate, research-backed information

Later as we proceed in the notebook, agent tools will query these databases to retrieve relevant information and help the LLM provide informed, grounded responses.


In [ ]:
# Loading pre-built datasets for web search and PubMed
with open('/Users/csharm33/code/genai_dbx/Course4/Data/search_data.json', 'r') as f:
    search_docs = json.load(f)

# Display the top-level keys in the loaded dataset
print(f"Major Document Types: {list(search_docs.keys())}")
tiktoken_cache_dir = os.path.abspath("./.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir
os.environ["ANONYMIZED_TELEMETRY"] = "False"

# Create a vector database from simulated web search documents.
# Enables semantic search over general health-related content.
web_search_db = Chroma.from_texts(
    search_docs['search_pages'],              # List of web-style document texts
    collection_name='web_search_db_demo4',    # Collection name
    embedding=embeddings_client,              # Embedding model for vectorization
)

# Create a vector database from simulated PubMed documents.
# Enables semantic search over clinical research content.
pubmed_db = Chroma.from_texts(
    search_docs['pubmed_pages'],              # List of PubMed-style document texts
    collection_name='pubmed_db_demo4',        # Collection name
    embedding=embeddings_client,              # Embedding model for vectorization
)


## Preparing Database for Doctor Recommendations Tool

In this section, we will create a small, in-memory database that contains information about doctors. This data will be used by our **Doctor Recommendation Tool**, which will help users find the right doctor based on their health query or symptoms.

The database includes a list of doctors along with their:
- Name
- Specialization (e.g., Dermatology, Pediatrics, Cardiology)
- Location
- Availability
- Contact information

We are using a simple Python list of dictionaries to store the doctor data. This keeps it easy to understand and modify. In a real-world application, this would typically be replaced by a backend database like PostgreSQL, MongoDB, or an external API.

We will build a tool later on to use this data to recommend doctors based on the user's needs — for example, recommending a pediatrician for a child’s fever or a cardiologist for chest pain.


In [ ]:
# Load doctor dataset with availability, specialization, and contact details
doctors_db = [
    {"name": "Dr. Janet Dyne", "specialization": "Endocrinology (Diabetes Care)", "available_timings": "10:00 AM - 1:00 PM", "location": "City Health Clinic", "contact": "janet.dyne@healthclinic.com"},
    {"name": "Dr. Don Blake", "specialization": "Cardiology (Heart Specialist)", "available_timings": "2:00 PM - 5:00 PM", "location": "Metro Cardiac Center", "contact": "don.blake@metrocardiac.com"},
    {"name": "Dr. Susan D'Souza", "specialization": "Oncology (Cancer Care)", "available_timings": "11:00 AM - 2:00 PM", "location": "Hope Cancer Institute", "contact": "susan.dsouza@hopecancer.org"},
    {"name": "Dr. Matt Murdock", "specialization": "Psychiatry (Mental Health)", "available_timings": "4:00 PM - 7:00 PM", "location": "Mind Care Center", "contact": "matt.murdock@mindcare.com"},
    {"name": "Dr. Dinah Lance", "specialization": "General Physician", "available_timings": "9:00 AM - 12:00 PM", "location": "Downtown Medical Center", "contact": "dinah.lance@downtownmed.com"}
]


## Preparing Database for Doctor Availability Slots Tool

In this section, we create a mock database that stores available appointment slots for each doctor. This data will be used by our `show_available_slots` tool to help users choose a time for booking.

We define a Python dictionary called `doctor_slots_db` where:
- Each doctor is a key
- The corresponding value is a dictionary with a key named `slots`, whose value is a list of available time slots for that doctor
- Each slot includes:
  - `time`: A string representing the time (e.g., "10:00 AM")
  - `available`: A boolean indicating whether the slot is currently free

This structure allows the agent to:
- Show open slots dynamically based on the user’s query
- Later update slot availability when a booking is made


In [ ]:
# Load the doctor appointment slots database with availability details
doctor_slots_db = {
    "Dr. Janet Dyne": {"slots": [
        {"time": "10:00 AM", "available": True}, {"time": "10:30 AM", "available": True},
        {"time": "11:00 AM", "available": True}, {"time": "11:30 AM", "available": True},
        {"time": "12:00 PM", "available": True}
    ]},
    "Dr. Don Blake": {"slots": [
        {"time": "2:00 PM", "available": True}, {"time": "2:30 PM", "available": True},
        {"time": "3:00 PM", "available": True}, {"time": "3:30 PM", "available": True},
        {"time": "4:00 PM", "available": True}
    ]},
    "Dr. Susan D'Souza": {"slots": [
        {"time": "11:00 AM", "available": True}, {"time": "11:30 AM", "available": True},
        {"time": "12:00 PM", "available": True}, {"time": "12:30 PM", "available": True},
        {"time": "1:00 PM", "available": True}
    ]},
    "Dr. Matt Murdock": {"slots": [
        {"time": "4:00 PM", "available": True}, {"time": "4:30 PM", "available": True},
        {"time": "5:00 PM", "available": True}, {"time": "5:30 PM", "available": True},
        {"time": "6:00 PM", "available": True}
    ]},
    "Dr. Dinah Lance": {"slots": [
        {"time": "9:00 AM", "available": True}, {"time": "9:30 AM", "available": True},
        {"time": "10:00 AM", "available": True}, {"time": "10:30 AM", "available": True},
        {"time": "11:00 AM", "available": True}
    ]}
}


## Preparing Database for Bookings Tool

This section defines a simple in-memory list called `bookings_db` that will store all appointment bookings made by the agent.

Each booking will be appended to this list with relevant details such as:
- Doctor name
- Chosen time slot
- User's name, email, and phone number

This data will be populated when the `book_appointment` tool is triggered by the agent.

The list is initially empty, and it will grow as users book appointments during their conversations with the agent. This lightweight structure helps simulate a real-time appointment booking system without needing a database.


In [ ]:
# Load the appointment bookings database — initially empty (no bookings yet)
bookings_db = []


## Create Tools for AI Agent

In this section, we define all the tools that our AI agent can use to perform specific actions while conversing with the user.

LangChain allows us to register these tools using the `@tool` decorator. Each tool consists of:
- A name and short description
- A Python function that implements the tool’s logic
- An input schema (inferred from the function signature) that guides the model on how to call it

These tools modularize complex behavior into callable units the agent can invoke when needed. When the LLM identifies a query that requires external data or a specific action, it can request a tool call with the correct arguments.

The tools defined here include:

- `search_web`: Retrieves general health-related information from the simulated web vector database
- `search_pubmed`: Looks up medical research content from the simulated PubMed vector store
- `recommend_doctor`: Suggests the most relevant doctor based on user symptoms using LLM reasoning
- `list_all_doctors`: Returns a list of all doctors and their specialties
- `show_available_slots`: Displays open appointment slots for a given doctor
- `book_appointment`: Books a slot for a user after collecting required details like doctor name, time, name, email, and phone and their slot of choice

All tools interact with in-memory data structures, making the setup lightweight and easy to experiment with (which you can easily switch out with your own databases if needed).

This tool-based design enables the ReAct loop — the agent can reason, decide, act using a tool, observe the result, and continue.


In [ ]:
# Tool for simulating a web search on general health topics
@tool
def search_web(query: str) -> list:
    """Search the web for a query. Useful for retrieving general or up-to-date healthcare information."""
    # Perform semantic similarity search over the web search vector database
    results = web_search_db.similarity_search(query, k=5)
    docs = [doc.page_content for doc in results]
    return docs


# Tool for simulating a PubMed-style academic search
@tool
def search_pubmed(query: str) -> list:
    """Search PubMed for scientific articles related to the query."""
    # Perform semantic similarity search over the PubMed vector database
    results = pubmed_db.similarity_search(query, k=5)
    docs = [doc.page_content for doc in results]
    return docs


# Tool for recommending a doctor based on user symptoms or query
@tool
def recommend_doctor(query: str) -> dict:
    """Recommend the most suitable doctor based on the user's symptoms."""
    doctors_list = str(doctors_db)

    # Use LLM reasoning to select the most appropriate doctor based on the query
    prompt = f"""You are an assistant helping recommend a doctor based on a patient's health issues.

Here is the list of available doctors:
{doctors_list}

Given the user's query: "{query}"

Choose the most suitable doctor from the list. Only pick one doctor.
Return only the selected doctor's information in JSON format.
If unsure, recommend the General Physician.
"""

    response = chat_client.invoke(prompt)
    return {"recommended_doctor": response.content}


# Tool to list all doctors in the clinic from the doctors database
@tool
def list_all_doctors() -> str:
    """Return the list of all doctors in the clinic along with their details."""
    return str(doctors_db)


# Tool to retrieve all available appointment slots for a specific doctor
@tool
def show_available_slots(doctor_name: str) -> list:
    """Retrieve and return available appointment slots for a given doctor."""
    if doctor_name not in doctor_slots_db:
        return ["Doctor not found."]

    # Filter for slots that are still available
    available = [slot["time"] for slot in doctor_slots_db[doctor_name]["slots"] if slot["available"]]
    return available or ["No available slots for this doctor."]


# Tool to book an appointment with a doctor based on patient details and selected time slot
@tool
def book_appointment(doctor_name: str, slot_time: str, patient_name: str, email: str, phone: str) -> str:
    """Book an appointment with a specific doctor for a selected time slot and store it in the bookings database."""

    # Check if the doctor exists in the slots database
    if doctor_name not in doctor_slots_db:
        return "Doctor not found."

    # Find and book the specified time slot if available
    for slot in doctor_slots_db[doctor_name]["slots"]:
        if slot["time"] == slot_time:
            if slot["available"]:
                slot["available"] = False  # Mark the slot as booked
                booking = {
                    "patient_name": patient_name,
                    "email": email,
                    "phone": phone,
                    "doctor_name": doctor_name,
                    "slot": slot_time
                }
                bookings_db.append(booking)
                return f"Appointment booked for {patient_name} with {doctor_name} at {slot_time}."
            else:
                return "The selected slot is no longer available."

    return "Requested slot not found for this doctor."


## Define Tools to be Used by the Agent

In this section, we define the list of tools that our AI agent will be allowed to use when responding to user queries.

These tools were previously defined using the `@tool` decorator. Here, we bundle them together and bind them to the LLM using LangChain’s `bind_tools()` method. This step is essential to give the model the ability to request tool calls dynamically based on the query.

The tools include:

- `search_web`: Fetches general health-related information from a simulated web document store
- `search_pubmed`: Retrieves scientific medical content from a simulated PubMed database
- `recommend_doctor`: Suggests an appropriate doctor based on user symptoms using LLM reasoning
- `list_all_doctors`: Lists all doctors and their specialties available in the clinic
- `show_available_slots`: Displays all currently available appointment slots for a specific doctor
- `book_appointment`: Books an appointment for the user after confirming the selected doctor, slot, and contact details


These tools are already defined and registered using the `@tool` decorator. Here, we simply collect them into a list and bind them to the LLM using LangChain’s `bind_tools()` method. This gives the LLM the ability to request tool usage during its reasoning process.

# List of all tools that the LLM should be aware of


In [ ]:
# These tools were defined earlier using the @tool decorator
tools = [
    search_web,
    search_pubmed,
    recommend_doctor,
    show_available_slots,
    book_appointment,
    list_all_doctors
]

# Bind the tools to the LLM so it can invoke tool calls when needed
# Enables tool-calling functionality in LangChain
llm_with_tools = chat_client.bind_tools(tools=tools)


## Define Agent Instructions Prompt

In this section, we define the system prompt that guides how the LLM-based agent should behave throughout the conversation. This instruction is key to ensuring the agent follows a structured flow, selects the right tools, and responds naturally and responsibly in a healthcare context.

The agent is instructed to act as:
- A medical research assistant that can look up health-related information
- A doctor recommender that can match user symptoms to a specialist
- A scheduling assistant that can help book doctor appointments

The system prompt covers specific behavior rules for each scenario:

### 🔍 Researching Health Information
- If the user query is about symptoms, treatments, or medical topics, use both `search_web` and `search_pubmed`
- If the query is general, `search_web` alone may be enough
- `search_pubmed` is used for more scientific or evidence-based queries
- Search results should include cited sources (links or publication metadata)

### 👨‍⚕️ Doctor Recommendation
- Use `recommend_doctor` only if the user is asking for a suggestion
- Use `list_all_doctors` if the user wants to browse all available doctors
- Present recommendations in a structured, readable format

### 🗓️ Viewing Available Slots
- First confirm the doctor exists using `list_all_doctors`
- If a valid doctor is matched, call `show_available_slots` and display open times
- If no match or no slots, respond gracefully and offer fallback options
- If based on a symptom, recommend a doctor first and then check their slots

### 📅 Booking Appointments
- Ensure the user has selected a valid slot (prompt again if not)
- Format time slots properly (e.g., `10:00 AM`, `3:30 PM`)
- Collect patient name, email, and phone before calling `book_appointment`
- Confirm the booking with a clear summary
- Avoid duplicate bookings if one is already confirmed

### ✅ General Guidelines
- Use tools when appropriate and return well-structured responses
- Decline politely if asked about topics unrelated to healthcare

This prompt allows the model to reason, reflect, decide which tool to call, observe the result, and then continue reasoning or respond based on ReAct.

It’s also tailored to handle multi-turn, real-world dialogue with multiple users.


In [ ]:
# Instruction prompt for the overall Agent
AGENT_PROMPT_TXT = """You are an agent designed to act as an expert in researching medical symptoms,
recommending relevant doctors for booking appointments, and assisting with the booking process.

Given a user query, call the relevant tools and provide the most appropriate response.
Follow these guidelines to make more informed decisions:

  - For researching information:
    - If the user is researching specific aspects such as symptoms, treatments, or other healthcare-related topics,
      use both search_web and search_pubmed to gather detailed information and provide a well-structured response.
    - If the user is looking for general healthcare information, search_web alone is sufficient.
    - Use search_pubmed only if the query relates to information likely found in PubMed articles.
    - The response should include cited source links from the web search and PubMed article titles and publication dates, if available.

  - For doctor recommendations:
    - If the user's query explicitly asks for a doctor recommendation, use the recommend_doctor tool.
    - If the user asks to see all available doctors, use the list_all_doctors tool and display the list.
    - When recommending a doctor, present the information in a clear, structured format.

  - For viewing doctor slots:
    - If the user wants to see slots for a specific doctor, follow this flow:
      - First, use the list_all_doctors tool to match the name provided by the user.
      - If no match is found, inform the user and show available doctors using the list_all_doctors tool.
        Ask if they would like to book an appointment with a specific doctor.
      - If a doctor match is found, use the show_available_slots tool and display the available slots in a well-structured format.
      - If no slots are available, apologize and ask the user to try again tomorrow.

    - If the user wants to see slots based on a symptom or problem, follow this flow:
      - First, use the recommend_doctor tool to identify the most relevant doctor and show the recommendation to the user.
        Ask if they would like to book an appointment.
      - If the doctor is accepted, use the show_available_slots tool and display the available slots in a well-structured format.
      - If no slots are available, apologize and ask the user to try again tomorrow.

  - For booking appointments:
    - If the user has already booked a slot, show the appointment details and avoid double booking.
    - If the user wants to book a specific recommended doctor, follow this flow:
      - First, get the exact name of the doctor (matching the name field) and use the show_available_slots tool
        to display available slots. Ask the user to choose a preferred slot.
      - If the slot is invalid, inform the user and ask them to enter a valid slot.
      - Ensure the slot time is properly formatted (e.g., 9:00 AM, 10:00 PM).
      - Then ask for the user's name, email, and phone number.
      - Use the book_appointment tool with the correct arguments to finalize the booking.
      - Present the confirmed appointment details in a clear, structured format.
      - If no slots are available, apologize and ask the user to try again tomorrow.

    - If the user has already viewed slots (as described above), follow this flow:
      - If the user hasn’t specified a preferred slot, prompt them to do so.
      - If the slot is invalid, explain why and request a valid one.
      - Ensure the slot time is formatted correctly (e.g., 9:00 AM, 10:00 PM).
      - Then ask for the user's name, email, and phone number.
      - Use the book_appointment tool with the appropriate arguments to book the appointment.
      - Display the confirmed appointment details in a structured format.
      - If no slots are available, apologize and ask the user to try again tomorrow.

  - Politely decline to answer any queries not related to medical or healthcare information.
"""

AGENT_SYS_PROMPT = SystemMessage(content=AGENT_PROMPT_TXT)


## Define Agent State Schema

In LangGraph, an agent’s state can be represented as a dictionary with various keys and values. Here we maintain state using a structured list of messages.

This state defines what data is carried forward between each step in the agent's reasoning and decision-making process.

Here, we define a simple state using Python’s `TypedDict` with one key field:

- `messages`: A list of all messages in the Agent so far. This includes the user query, LLM responses, any tool call requests, and the tool responses

The `messages` list is annotated using `add_messages`, which ensures that every new message is appended properly as the agent moves through each step in the graph.

This schema is later passed to the LangGraph `StateGraph`, enabling the agent to track all reasoning and tool usage as a growing message history.

It forms the foundation for the ReAct loop — Reason, Act (tool call), Observe, Repeat — that we build using LangGraph.


In [ ]:
# Define the agent's state schema for storing the message history
class State(TypedDict):
    messages: Annotated[list, add_messages]


## Define Tool-Use Conversational Agent with LangGraph

In this section, we define our full conversational agent using LangGraph’s `StateGraph`, combining reasoning, tool execution, and memory support.

This agent follows the ReAct pattern — it thinks through the query, calls tools if needed, observes results, and continues reasoning or responds directly.

### 🧠 Key Components of the Agent:

- **LLM Reasoning Node (`tool_calling_llm`)**:
  The agent uses a planning function where the LLM analyzes the current message history (including prior tool calls and responses) and decides what to do next

- **Tool Node (`ToolNode`)**:
  When the LLM requests a tool call, this node executes the right tool based on the request (e.g., search, recommend, view slots, or book)

- **Routing Logic**:
  Using `add_conditional_edges()`, the system checks if the LLM output is a tool call or a final response:
  - If it's a tool call, it routes to the `tools` node
  - If not, it ends and returns the final output

- **Looping**:
  After tool execution, the agent feeds the output back into the message history and re-enters the planning node. This loop continues until a final response is ready

### 💬 Conversational Memory:

To support multi-turn conversations and track context, the agent uses:
```python


In [ ]:
# Create the node function that handles reasoning and planning using the LLM
def tool_calling_llm(state: State) -> State:
    # Extract the current conversation history from the state
    current_state = state["messages"]

    # Trim the message history to fit within the model's token limit
    trimmed_state = trim_messages(
        state["messages"],
        max_tokens=127000,              # GPT-4o-mini supports up to ~128K tokens
        strategy="last",                # Retain the most recent messages
        token_counter=count_tokens_approximately,  # Use approximate token counting
        include_system=True,            # Keep the system prompt intact
        allow_partial=True              # Allow partial trimming of messages if needed
    )

    # Prepend the system instruction to the trimmed message history
    state_with_instructions = [AGENT_SYS_PROMPT] + trimmed_state

    # Call the LLM to generate a new message (either a direct response or a tool call)
    response = [llm_with_tools.invoke(state_with_instructions)]

    # Return the updated state including the new message
    return {"messages": response}


# Build the agent execution graph
builder = StateGraph(State)

# Add nodes to the graph
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

# Add edges
builder.add_edge(START, "tool_calling_llm")

# Conditional routing:
# - If the latest message includes a tool call -> route to "tools"
# - Otherwise -> end the workflow
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,  # Routing function based on the message
    ["tools", END]
)

# Feedback loop: return to LLM for reasoning after tool execution
builder.add_edge("tools", "tool_calling_llm")

# Compile the agent graph and attach conversational memory for persistence
memory = MemorySaver()
healthbuddy_agent = builder.compile(checkpointer=memory)


## View our Agent Flow

This is what the agent workflow looks like.


In [ ]:
from IPython.display import Image
Image(filename='/Users/csharm33/code/genai_dbx/Course4/Data/tool_use_agent_arch.png', height=200, width=300)


## Build Utility Function to Stream Conversational Agent Results

In this section, we define a helper function to run the agent and stream its intermediate reasoning steps as it processes a user query.

This is especially useful for:
- Observing what the agent is doing at each step (thinking, acting, observing)
- Seeing which tools it decides to call, if any
- Viewing the raw outputs of those tools
- Watching how it uses those observations to form the final response

The function also supports:
- Multi-user sessions using a `user_session_id`, which links each query to the appropriate memory
- Verbose mode for printing detailed agent output after every step
- Nicely formatted final output using Markdown for improved readability in notebooks

This utility is helpful during development and debugging to understand the internal workings of the ReAct loop and tool invocation logic.


In [ ]:
# Utility function to call the agent and stream its step-by-step reasoning for a specific user
def call_conversational_agent(agent, prompt, user_session_id, verbose=False):
    # Stream the agent's execution using the given prompt and session ID for memory tracking
    events = agent.stream(
        {"messages": [{"role": "user", "content": prompt}]},   # User input prompt
        {"configurable": {"thread_id": user_session_id}},      # Thread ID for session-based memory (multi-user support)
        stream_mode="values"                                   # Stream messages as the agent processes them
    )

    print('Running Agent. Please wait...')

    # Iterate through each step of the streamed agent output
    for event in events:
        # If verbose mode is enabled, print each intermediate message
        if verbose:
            event["messages"][-1].pretty_print()

    # Display the final response from the agent as formatted Markdown
    print('\n\nAgent Final Response:\n')
    display(Markdown(event["messages"][-1].content))


## Test out our Conversational Agent!

In this final section, we run a complete test of our tool-using conversational agent by sending it queries from different users.

We observe how the agent:
- Interprets the user query
- Decides which tool(s) to use (if any)
- Executes the requested tool and integrates its output
- Streams its intermediate thoughts and reasoning steps
- Returns a well-structured, context-aware response

### 💬 Multi-User Session Testing

We simulate two different users interacting with the agent using unique `user_session_id`s. This demonstrates:
- How the agent maintains separate memory per user
- How each conversation progresses independently
- That prior messages, tool responses, and context are remembered for each user individually

This test helps validate that the agent behaves like a real-world healthcare assistant — capable of:
- Multi-turn dialogue
- Dynamic tool use
- Personalized interactions for multiple users in parallel


### Simulating a Conversation Between User 1 and the Agent

Each user has a unique user ID; in this case, the ID is `john001`.


In [ ]:
uid = 'john001'
query = 'what doctors are available?'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'I need to see a doctor for panic attacks'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)

query = 'yes please I would like to book an appointment immediately'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'the 4 pm slot would be good'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'John, john@email.com, 1001099'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)


You can see that the bookings database now has a new entry made autonomously by the agent
bookings_db


You can also see the 4 pm slot for Dr. Matt has been now set to False (unavailable) by the agent
doctor_slots_db['Dr. Matt Murdock']


### Simulating a Conversation Between User 2 and the Agent

Each user has a unique user ID; in this case, the ID is `jim007`.


In [ ]:
uid = 'jim007'
query = 'having a lot of stress and depression, what are some ways to tackle it? and also recommend a doctor for me'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'yes please'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = '4 pm please'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'no can I see the available slots again'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'lets do 6'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)
query = 'jim, jim@email.com, 10201221'
call_conversational_agent(agent=healthbuddy_agent,
                          prompt=query,
                          user_session_id=uid,
                          verbose=True)


You can see that the bookings database now has a new entry made autonomously by the agent
bookings_db


You can also see the 6:00 pm slot for Dr. Matt has been now set to False (unavailable) by the agent
doctor_slots_db['Dr. Matt Murdock']


## Homework: Add an Insurance Eligibility Checker Tool to the Multi-User HealthBuddy Agent
**Goal:** Enhance the HealthBuddy conversational agent by implementing a new `insurance_eligibility_checker` tool that simulates checking whether a medical treatment or procedure is covered by a user’s insurance. This tool should follow the same pattern as other tools in the notebook and integrate seamlessly into the LangGraph-based agent.


#### Tasks
1. **Define a Static Coverage Dataset:**  Create a Python dictionary that maps sample treatments to insurance coverage responses.
You can use this sample data and add more examples to experiment:
![image.png](attachment:fffa69d4-886d-4520-a8b4-771e6b5aef6f.png)

2. **Implement the Tool Function:** Define a tool using the `@tool` decorator. Instead of returning directly from the dictionary using hardcoded keyword or string matching, use the LLM to reason and return the best response, similar to how the `recommend_doctor` tool is implemented.
![image.png](attachment:e2d2283c-84bb-4c83-b195-02bec86980ec.png)
   

3. **Update the System Instruction Prompt:** Add guidance to the system prompt describing what the new tool does and in what kinds of user queries it should be considered.

4. **Register the Tool in the Agent Configuration:** Add the tool to the tools list passed to `create_react_agent()`. Ensure it’s part of the tool-using ReAct loop like the other tools.

5. **Simulate Multi-User Scenarios:** Test with two different users asking about coverage:
    - User A: “Will insurance cover a flu shot?”
    - User B: “Is an MRI scan included in my health plan?”
6. **Check that:**
    - Each user gets the correct tool-based response
    - Memory is isolated and working independently per user session
